# Pruebas de metricas -- Entregable 2

Prototipado iterativo (paso a paso, confirmado por el usuario antes de
avanzar) de la actividad (b) del TdR. Ver
`informe/borradores/marco_teorico_e2.txt` para las ecuaciones y
referencias -- cada seccion cita el numero de seccion de ese
documento.

**Orden metodologico** (decision del usuario): climatologia y sesgo
primero (son el insumo de la correccion), despues Linear Scaling (LS),
y RECIEN DESPUES todo lo demas -- RMSE, correlacion, variabilidad,
diagrama de Taylor, ONI/RONI/ICEN -- se calcula sobre la serie ya
corregida, nunca sobre la serie cruda del modelo. LS es una correccion
aditiva constante por mes calendario, asi que no cambia la
variabilidad/correlacion/RMSE centrado (son invariantes a un corrimiento
constante) -- lo que si cambia es el RMSE total (ya no mezcla el sesgo
medio con el error de variabilidad). El orden del notebook igual queda
asi de punta a punta, por consistencia metodologica.

Una funcion por calculo, probada primero con un subconjunto chico de
modelos (via `glob`), no los 40 completos -- recien cuando todo este
validado se convierte en `.py` (funciones + un ejecutor), en
`scripts_e2/`.

No se versiona en GitHub (`.gitignore`), igual que
`scripts/graficos_exploratorios.ipynb`.

In [1]:
import glob
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
MASKED = ROOT / "data/processed/masked"
INTERIM = ROOT / "data/interim/processed"
OUT_FIG = ROOT / "figures/e2"
OUT_FIG.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
import plot_common as pc

NINO34 = pc.NINO34
NINO12 = pc.NINO12
TROPICAL = dict(lon1=0, lon2=360, lat1=-20, lat2=20)  # RONI: Trenberth 1997 / convencion NOAA CPC
REF_INICIO, REF_FIN = 1981, 2014

N_TEST = 4
_masked_hist = sorted(MASKED.glob("tos_*_historical.nc"))
_masked_ready = (MASKED / "ersstv5_region.nc").exists() and len(_masked_hist) >= N_TEST
DATA_DIR = MASKED if _masked_ready else INTERIM
if DATA_DIR != MASKED:
    print(f"AVISO: {MASKED} incompleto todavia -- usando {DATA_DIR} (sin mascara oceano-tierra) para prototipar.")

print(NINO34, NINO12, TROPICAL)

{'lon1': 190, 'lon2': 240, 'lat1': -5, 'lat2': 5} {'lon1': 270, 'lon2': 280, 'lat1': -10, 'lat2': 0} {'lon1': 0, 'lon2': 360, 'lat1': -20, 'lat2': 20}


## 0. Seleccion de modelos de prueba (glob, subconjunto chico)

In [2]:
hist_files = sorted(glob.glob(str(DATA_DIR / "tos_*_historical.nc")))
all_models = [Path(f).stem.replace("tos_", "").replace("_historical", "") for f in hist_files]
test_models = all_models[:N_TEST]
print(f"{len(all_models)} modelos disponibles en {DATA_DIR.name}, probando con {N_TEST}: {test_models}")

40 modelos disponibles en masked, probando con 4: ['ACCESS-CM2', 'ACCESS-ESM1-5', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR']


## 1. Series por caja (obs + modelos de prueba)

Promedio espacial simple (Nino3.4/Nino1+2, cajas angostas cerca del
ecuador) o ponderado por cos(lat) (banda TROPICAL, mucho mas ancha).
Tiempo normalizado a dia-1-del-mes: ERSSTv5 y los modelos CMIP6 no
comparten convencion de dia (1 vs. 15/16), y sin esto cualquier
comparacion directa (`xr.align`) da una interseccion vacia -- NaN
silencioso en RMSE/correlacion/Taylor mas adelante.

In [3]:
def _select_box(da, box):
    sub = da.sel(lat=slice(box["lat1"], box["lat2"]), lon=slice(box["lon1"], box["lon2"]))
    if sub.lat.size == 0:
        sub = da.sel(lat=slice(box["lat2"], box["lat1"]), lon=slice(box["lon1"], box["lon2"]))
    return sub


def _normalize_time(da):
    """Fuerza el tiempo a Timestamp de pandas, dia-1-del-mes, sin
    importar el calendario nativo del archivo. Varios modelos CMIP6
    usan calendarios no estandar (noleap, 360_day, etc.) -- xarray los
    decodifica como CFTimeIndex en vez de DatetimeIndex de pandas, que
    no tiene .to_period(). Peor: aunque se evite ese metodo, comparar
    fechas cftime contra Timestamps de pandas (ERSSTv5, calendario
    estandar) con xr.align tampoco alinea -- son tipos distintos aunque
    "sea la misma" fecha. Se reconstruye la fecha como texto desde
    .year/.month (atributos que tanto cftime como pandas tienen) para
    que TODO quede en un Timestamp de pandas uniforme, comparable entre
    modelos sin importar su calendario de origen."""
    idx = da.indexes["time"]
    nuevas = pd.to_datetime([f"{t.year:04d}-{t.month:02d}-01" for t in idx])
    return da.assign_coords(time=nuevas)


def box_series(path, box, varname, weighted=False):
    ds = xr.open_dataset(path)
    sub = _select_box(ds[varname], box)
    if not weighted:
        out = sub.mean(dim=("lat", "lon"))
    else:
        w = np.cos(np.deg2rad(sub.lat))
        out = sub.weighted(w).mean(dim=("lat", "lon"))
    return _normalize_time(out)


def model_path(model, exp="historical"):
    return DATA_DIR / f"tos_{model}_{exp}.nc"

In [4]:
BOXES = {"nino34": NINO34, "nino12": NINO12, "tropical": TROPICAL}
WEIGHTED = {"nino34": False, "nino12": False, "tropical": True}

obs_path = DATA_DIR / "ersstv5_region.nc"
obs_series = {k: box_series(obs_path, box, "sst", weighted=WEIGHTED[k]) for k, box in BOXES.items()}

model_series = {
    m: {k: box_series(model_path(m), box, "tos", weighted=WEIGHTED[k]) for k, box in BOXES.items()}
    for m in test_models
}

obs_series["nino34"].sel(time=slice("1981", "2014")).plot()
plt.title("ERSSTv5 -- Nino 3.4 (sanity check, periodo de referencia)")
plt.savefig(OUT_FIG / "_test_sanity_obs_nino34.png", dpi=100, bbox_inches="tight")
plt.show()

/tmp/ipykernel_40441/3599923550.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Climatologia y ciclo anual (Seccion 1 del marco teorico)

$C_m = \frac{1}{N}\sum_{y} X(y,m)$, sobre 1981-2014.

In [5]:
def climatology(da, y0=REF_INICIO, y1=REF_FIN):
    ref = da.sel(time=slice(f"{y0}", f"{y1}"))
    return ref.groupby("time.month").mean("time")

clim_obs = {k: climatology(v) for k, v in obs_series.items()}
clim_model = {m: {k: climatology(model_series[m][k]) for k in BOXES} for m in test_models}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, box_name, key in zip(axes, ["Nino 3.4", "Nino 1+2"], ["nino34", "nino12"]):
    ax.plot(clim_obs[key].month, clim_obs[key], "k-", lw=2.5, label="ERSSTv5")
    for m in test_models:
        ax.plot(clim_model[m][key].month, clim_model[m][key], alpha=0.8, label=m)
    ax.set_title(f"Ciclo anual -- {box_name} (1981-2014)")
    ax.set_xlabel("Mes")
    ax.set_ylabel("SST (degC)")
    ax.legend(fontsize=7)
plt.savefig(OUT_FIG / "_test_ciclo_anual.png", dpi=100, bbox_inches="tight")
plt.show()

/tmp/ipykernel_40441/2898069070.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Sesgo (Seccion 3): $B_m = C_m^{modelo} - C_m^{obs}$

Insumo directo de la correccion LS de la Seccion 4 -- se calcula antes,
sobre el modelo crudo, a proposito.

In [6]:
def bias_monthly(clim_model_da, clim_obs_da):
    return clim_model_da - clim_obs_da

bias = {m: {k: bias_monthly(clim_model[m][k], clim_obs[k]) for k in BOXES} for m in test_models}

rows = [{"modelo": m, "caja": k, "sesgo_medio_degC": float(bias[m][k].mean())}
        for m in test_models for k in BOXES]
pd.DataFrame(rows)

,modelo,caja,sesgo_medio_degC
0,ACCESS-CM2,nino34,-0.017585
1,ACCESS-CM2,nino12,3.113023
2,ACCESS-CM2,tropical,0.097037
3,ACCESS-ESM1-5,nino34,0.102361
4,ACCESS-ESM1-5,nino12,2.933653
5,ACCESS-ESM1-5,tropical,0.347361
6,AWI-CM-1-1-MR,nino34,-1.413901
7,AWI-CM-1-1-MR,nino12,1.328486
8,AWI-CM-1-1-MR,tropical,-0.256484
9,BCC-CSM2-MR,nino34,-0.071285


## 4. Correccion de sesgo por Linear Scaling -- LS (Seccion 8)

$X^*_{modelo}(y,m) = X_{modelo}(y,m) - B_m$

**A partir de aca, todo (anomalias, RMSE, correlacion, variabilidad,
Taylor, ONI/RONI/ICEN) se calcula sobre `corrected`, nunca sobre
`model_series` crudo.**

In [7]:
def linear_scaling(model_da, bias_da):
    return model_da.groupby("time.month") - bias_da

corrected = {
    m: {k: linear_scaling(model_series[m][k], bias[m][k]) for k in BOXES}
    for m in test_models
}

for m in test_models:
    b_after = bias_monthly(climatology(corrected[m]["nino34"]), clim_obs["nino34"])
    print(f"{m:16s} sesgo Nino3.4 tras LS (deberia ser ~0): {float(b_after.mean()):.2e}")

ACCESS-CM2       sesgo Nino3.4 tras LS (deberia ser ~0): -7.11e-07
ACCESS-ESM1-5    sesgo Nino3.4 tras LS (deberia ser ~0): -7.57e-07
AWI-CM-1-1-MR    sesgo Nino3.4 tras LS (deberia ser ~0): 5.24e-07
BCC-CSM2-MR      sesgo Nino3.4 tras LS (deberia ser ~0): 4.07e-07


## 5. Anomalias (Seccion 2)

$A(y,m) = X^*(y,m) - C_m^{obs}$ -- sobre la serie ya corregida (su
climatologia ya coincide con la observada, Seccion 4).

In [8]:
def anomaly(da, clim):
    return da.groupby("time.month") - clim

anom_obs = {k: anomaly(v, clim_obs[k]) for k, v in obs_series.items()}
anom_model = {m: {k: anomaly(corrected[m][k], clim_obs[k]) for k in BOXES} for m in test_models}
print("anomalia obs Nino3.4, primeros 3 valores:", anom_obs["nino34"].values[:3])

anomalia obs Nino3.4, primeros 3 valores: [-0.55893135 -0.61236572 -0.28153229]


## 6. RMSE (Seccion 4): total y centrado, sobre la serie corregida

El RMSE centrado no cambia respecto a la serie cruda (LS es un
corrimiento constante por mes, no afecta la dispersion) -- el que si
cambia es el total: ya no mezcla el sesgo medio con el error de
variabilidad.

In [9]:
def rmse(model_da, obs_da, centered=False):
    m, o = xr.align(model_da, obs_da, join="inner")
    if centered:
        m = m - m.mean()
        o = o - o.mean()
    return float(np.sqrt(((m - o) ** 2).mean()))

for m in test_models:
    r34 = rmse(corrected[m]["nino34"], obs_series["nino34"])
    r34c = rmse(corrected[m]["nino34"], obs_series["nino34"], centered=True)
    print(f"{m:16s} Nino3.4  RMSE={r34:.3f}  RMSE'(centrado)={r34c:.3f}")

ACCESS-CM2       Nino3.4  RMSE=1.185  RMSE'(centrado)=1.178
ACCESS-ESM1-5    Nino3.4  RMSE=1.192  RMSE'(centrado)=1.192
AWI-CM-1-1-MR    Nino3.4  RMSE=1.327  RMSE'(centrado)=1.327
BCC-CSM2-MR      Nino3.4  RMSE=1.194  RMSE'(centrado)=1.192


## 7. Correlacion de Pearson (Seccion 5), sobre la serie corregida

In [10]:
def pearson_r(model_da, obs_da):
    m, o = xr.align(model_da, obs_da, join="inner")
    return float(xr.corr(m, o))

for m in test_models:
    print(f"{m:16s} Nino3.4  r={pearson_r(corrected[m]['nino34'], obs_series['nino34']):.3f}")

ACCESS-CM2       Nino3.4  r=0.261
ACCESS-ESM1-5    Nino3.4  r=0.278
AWI-CM-1-1-MR    Nino3.4  r=0.181
BCC-CSM2-MR      Nino3.4  r=0.371


## 8. Variabilidad interanual (Seccion 7): $\sigma$ de las anomalias (ya sobre la serie corregida)

In [11]:
def interannual_std(anomaly_da):
    return float(anomaly_da.std(ddof=1))

sigma_obs_34 = interannual_std(anom_obs["nino34"])
print(f"sigma obs Nino3.4 = {sigma_obs_34:.3f}")
for m in test_models:
    s = interannual_std(anom_model[m]["nino34"])
    print(f"{m:16s} sigma={s:.3f}  (sigma_modelo/sigma_obs = {s/sigma_obs_34:.2f})")

sigma obs Nino3.4 = 0.898
ACCESS-CM2       sigma=0.824  (sigma_modelo/sigma_obs = 0.92)
ACCESS-ESM1-5    sigma=0.865  (sigma_modelo/sigma_obs = 0.96)
AWI-CM-1-1-MR    sigma=0.962  (sigma_modelo/sigma_obs = 1.07)
BCC-CSM2-MR      sigma=1.027  (sigma_modelo/sigma_obs = 1.14)


## 9. Diagrama de Taylor -- estadisticos (Seccion 6), sobre la serie corregida

$E'^2 = \sigma_f^2 + \sigma_r^2 - 2\sigma_f\sigma_r r$.

In [12]:
def taylor_stats(model_da, obs_da):
    m, o = xr.align(model_da, obs_da, join="inner")
    sigma_f = float(m.std(ddof=1))
    sigma_r = float(o.std(ddof=1))
    r = pearson_r(m, o)
    e_prime = rmse(m, o, centered=True)
    return dict(sigma_f=sigma_f, sigma_r=sigma_r, r=r, rmse_centrado=e_prime, sigma_norm=sigma_f / sigma_r)

rows = []
for m in test_models:
    stats = taylor_stats(anom_model[m]["nino34"], anom_obs["nino34"])
    rows.append({"modelo": m, **stats})
pd.DataFrame(rows)

,modelo,sigma_f,sigma_r,r,rmse_centrado,sigma_norm
0,ACCESS-CM2,0.826547,0.880131,0.046898,1.178498,0.939118
1,ACCESS-ESM1-5,0.869900,0.880131,0.071955,1.191822,0.988375
2,AWI-CM-1-1-MR,0.966298,0.880131,-0.030764,1.326567,1.097902
3,BCC-CSM2-MR,1.030429,0.880131,0.228041,1.192499,1.170767


## 10. ONI y RONI -- Nino 3.4 (Seccion 9), sobre la serie corregida

ONI = media movil de 3 meses de la anomalia; RONI = ONI menos la
anomalia de la TSM media tropical (tambien corregida por LS).

In [13]:
def rolling3(da):
    return da.rolling(time=3, center=True).mean()

def oni_index(corrected_nino34_da, clim_da):
    a = anomaly(corrected_nino34_da, clim_da)
    return rolling3(a)

def roni_index(oni_da, corrected_tropical_da, clim_tropical_da):
    """RONI = ONI - anomalia tropical (rolling 3m), reescalado para que
    su varianza iguale a la del ONI original -- definicion oficial NOAA
    CPC (cpc.ncep.noaa.gov/products/analysis_monitoring/enso/roni/),
    confirmada por busqueda web. Faltaba este reescalado en la primera
    version de esta funcion: sin el, RONI queda sistematicamente con
    menos varianza que ONI, no comparable en amplitud."""
    a_trop = anomaly(corrected_tropical_da, clim_tropical_da)
    raw = oni_da - rolling3(a_trop)
    m_oni, m_raw = xr.align(oni_da, raw, join="inner")
    scale = float(m_oni.std(ddof=1) / m_raw.std(ddof=1))
    return raw * scale

oni_obs = oni_index(obs_series["nino34"], clim_obs["nino34"])
roni_obs = roni_index(oni_obs, obs_series["tropical"], clim_obs["tropical"])

oni_model, roni_model = {}, {}
for m in test_models:
    oni_model[m] = oni_index(corrected[m]["nino34"], clim_obs["nino34"])
    roni_model[m] = roni_index(oni_model[m], corrected[m]["tropical"], clim_obs["tropical"])

print(f"Verificacion reescalado -- std(ONI obs)={float(oni_obs.std()):.3f}  std(RONI obs)={float(roni_obs.std()):.3f} (deberian ser iguales)")

fig, ax = plt.subplots(figsize=(11, 3.5))
oni_obs.sel(time=slice("1990", "2010")).plot(ax=ax, color="k", lw=1.8, label="ONI obs")
for m in test_models:
    oni_model[m].sel(time=slice("1990", "2010")).plot(ax=ax, alpha=0.8, label=f"ONI {m} (LS)")
ax.axhline(0.5, color="r", ls="--", lw=0.7)
ax.axhline(-0.5, color="b", ls="--", lw=0.7)
ax.legend(fontsize=6, ncol=2)
ax.set_title("ONI (corregido por LS) -- ventana 1990-2010")
plt.savefig(OUT_FIG / "_test_oni.png", dpi=100, bbox_inches="tight")
plt.show()

Verificacion reescalado -- std(ONI obs)=0.868  std(RONI obs)=0.868 (deberian ser iguales)


/tmp/ipykernel_40441/4002921242.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, ax = plt.subplots(figsize=(11, 3.5))
roni_obs.sel(time=slice("1990", "2010")).plot(ax=ax, color="k", lw=1.8, label="RONI obs")
for m in test_models:
    roni_model[m].sel(time=slice("1990", "2010")).plot(ax=ax, alpha=0.8, label=f"RONI {m} (LS)")
ax.axhline(0.5, color="r", ls="--", lw=0.7)
ax.axhline(-0.5, color="b", ls="--", lw=0.7)
ax.legend(fontsize=7)
ax.set_title("RONI (ONI - TSM media tropical, corregido por LS) -- ventana 1990-2010")
plt.savefig(OUT_FIG / "_test_roni.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"RONI obs: media={float(roni_obs.mean()):.3f}  std={float(roni_obs.std()):.3f}")
for m in test_models:
    r = roni_model[m]
    print(f"RONI {m}: media={float(r.mean()):.3f}  std={float(r.std()):.3f}")

RONI obs: media=-0.011  std=0.868
RONI ACCESS-CM2: media=0.012  std=0.799
RONI ACCESS-ESM1-5: media=-0.116  std=0.843
RONI AWI-CM-1-1-MR: media=-0.039  std=0.933
RONI BCC-CSM2-MR: media=-0.142  std=0.999


/tmp/ipykernel_40441/2258482822.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. ICEN -- Nino 1+2 (Seccion 10), sobre la serie corregida

In [15]:
def icen_index(corrected_nino12_da, clim_da):
    a = anomaly(corrected_nino12_da, clim_da)
    return rolling3(a)

icen_obs = icen_index(obs_series["nino12"], clim_obs["nino12"])
icen_model = {m: icen_index(corrected[m]["nino12"], clim_obs["nino12"]) for m in test_models}

fig, ax = plt.subplots(figsize=(11, 3.5))
icen_obs.sel(time=slice("1990", "2010")).plot(ax=ax, color="k", lw=1.8, label="ICEN obs")
for m in test_models:
    icen_model[m].sel(time=slice("1990", "2010")).plot(ax=ax, alpha=0.8, label=f"ICEN {m} (LS)")
ax.axhline(0.4, color="r", ls="--", lw=0.7)
ax.legend(fontsize=7)
ax.set_title("ICEN (corregido por LS) -- ventana 1990-2010")
plt.savefig(OUT_FIG / "_test_icen.png", dpi=100, bbox_inches="tight")
plt.show()

/tmp/ipykernel_40441/3425435214.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Clasificacion de eventos calidos extremos (Seccion 11, punto 4)

Umbrales *pendientes de confirmar*: ONI >= +0.5 degC por >= 5 estaciones
de 3 meses; ICEN >= +0.4 degC por >= 3 meses. Parametrizado, no
hardcodeado.

In [16]:
def classify_events(index_da, threshold=0.5, min_consecutive=5):
    vals = index_da.values
    times = index_da["time"].values
    events = []
    i = 0
    n = len(vals)
    while i < n:
        if np.isfinite(vals[i]) and vals[i] >= threshold:
            j = i
            while j < n and np.isfinite(vals[j]) and vals[j] >= threshold:
                j += 1
            if (j - i) >= min_consecutive:
                events.append((times[i], times[j - 1], float(np.nanmax(vals[i:j]))))
            i = j
        else:
            i += 1
    return events

print("Eventos ONI obs (>=0.5, >=5 meses):")
for ev in classify_events(oni_obs, threshold=0.5, min_consecutive=5):
    print(f"  {pd.Timestamp(ev[0]).date()} a {pd.Timestamp(ev[1]).date()}, pico={ev[2]:.2f}")

print("\nEventos ICEN obs (>=0.4, >=3 meses):")
for ev in classify_events(icen_obs, threshold=0.4, min_consecutive=3):
    print(f"  {pd.Timestamp(ev[0]).date()} a {pd.Timestamp(ev[1]).date()}, pico={ev[2]:.2f}")

Eventos ONI obs (>=0.5, >=5 meses):
  1855-10-01 a 1856-02-01, pico=1.03
  1877-06-01 a 1878-05-01, pico=2.54
  1888-08-01 a 1889-04-01, pico=2.21
  1896-07-01 a 1897-02-01, pico=1.56
  1899-11-01 a 1900-06-01, pico=1.24
  1902-05-01 a 1903-03-01, pico=1.71
  1904-12-01 a 1906-01-01, pico=1.10
  1911-11-01 a 1912-04-01, pico=1.07
  1914-08-01 a 1915-05-01, pico=1.14
  1918-10-01 a 1919-02-01, pico=1.58
  1925-11-01 a 1926-04-01, pico=1.12
  1930-09-01 a 1931-03-01, pico=1.10
  1940-08-01 a 1941-07-01, pico=1.96
  1957-04-01 a 1958-04-01, pico=1.59
  1963-08-01 a 1964-01-01, pico=0.92
  1965-07-01 a 1966-03-01, pico=1.46
  1968-12-01 a 1969-04-01, pico=0.99
  1972-06-01 a 1973-02-01, pico=1.99
  1976-10-01 a 1977-02-01, pico=0.73
  1982-05-01 a 1983-06-01, pico=2.23
  1986-09-01 a 1988-02-01, pico=1.55
  1991-06-01 a 1992-06-01, pico=1.83
  1994-09-01 a 1995-03-01, pico=1.20
  1997-05-01 a 1998-04-01, pico=2.43
  2002-06-01 a 2003-02-01, pico=1.42
  2004-07-01 a 2005-02-01, pico=0.79
  

## 13. Resumen: diagrama de Taylor por indice (Seccion 11, punto 3)

Aplica `taylor_stats` (Seccion 9) a cada indice (ONI y RONI en Nino3.4,
ICEN en Nino1+2) comparando modelo (corregido por LS) vs. ERSSTv5.

In [17]:
rows = []
for m in test_models:
    for label, idx_model, idx_obs in [
        ("ONI (Nino3.4)", oni_model[m], oni_obs),
        ("RONI (Nino3.4)", roni_model[m], roni_obs),
        ("ICEN (Nino1+2)", icen_model[m], icen_obs),
    ]:
        stats = taylor_stats(idx_model, idx_obs)
        rows.append({"modelo": m, "indice": label, **stats})

summary = pd.DataFrame(rows)
summary

,modelo,indice,sigma_f,sigma_r,r,rmse_centrado,sigma_norm
0,ACCESS-CM2,ONI (Nino3.4),0.802223,0.851411,0.049976,1.139954,0.942227
1,ACCESS-CM2,RONI (Nino3.4),0.802617,0.861614,0.012928,1.169830,0.931526
2,ACCESS-CM2,ICEN (Nino1+2),0.678519,0.972578,0.009525,1.180431,0.697650
3,ACCESS-ESM1-5,ONI (Nino3.4),0.848403,0.851411,0.076700,1.154684,0.996467
4,ACCESS-ESM1-5,RONI (Nino3.4),0.848443,0.861614,0.016657,1.199053,0.984713
5,ACCESS-ESM1-5,ICEN (Nino1+2),0.866224,0.972578,0.067878,1.257604,0.890648
6,AWI-CM-1-1-MR,ONI (Nino3.4),0.936654,0.851411,-0.029246,1.283820,1.100119
7,AWI-CM-1-1-MR,RONI (Nino3.4),0.938686,0.861614,-0.086884,1.328112,1.089450
8,AWI-CM-1-1-MR,ICEN (Nino1+2),0.849875,0.972578,-0.044674,1.319747,0.873838
9,BCC-CSM2-MR,ONI (Nino3.4),1.001867,0.851411,0.236524,1.150894,1.176713


## 14. Diagrama de Taylor -- grafico (no solo la tabla de estadisticos)

Estandar en la literatura de evaluacion CMIP6/ENOS (ver p.ej. "CMIP6 model evaluation for sea surface height responses to ENSO", Climate Dynamics, 10.1007/s00382-023-06997-z, que usa Taylor diagram sobre Nino3.4). Colores categoricos en orden fijo (tab10, hasta 10 modelos) -- un modelo, un color, consistente en todos los paneles. Punto de referencia (ERSSTv5) en (1, 0).

In [18]:
MODEL_COLORS = {m: plt.cm.tab10(i % 10) for i, m in enumerate(test_models)}


def plot_taylor_diagram(ax, stats_rows, title):
    """stats_rows: lista de dicts con 'modelo', 'sigma_norm', 'r'.
    Grafico polar-cartesiano estandar (Taylor, 2001): angulo=arccos(r),
    radio=sigma_norm. Referencia (obs) en (1,0)."""
    max_r = max([row["sigma_norm"] for row in stats_rows] + [1.3]) * 1.15

    theta_grid = np.linspace(0, np.pi / 2, 200)
    for rc in np.arange(0.25, max_r, 0.25):
        ax.plot(rc * np.cos(theta_grid), rc * np.sin(theta_grid), color="0.88", lw=0.6, zorder=0)
    for corr_val in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
        ang = np.arccos(corr_val)
        ax.plot([0, max_r * np.cos(ang)], [0, max_r * np.sin(ang)], color="0.92", lw=0.5, zorder=0)
        ax.text(max_r * 1.01 * np.cos(ang), max_r * 1.01 * np.sin(ang), f"{corr_val:g}",
                fontsize=6, color="0.5", ha="left", va="bottom")

    ax.plot([1], [0], marker="*", color="k", markersize=15, zorder=5, linestyle="none", label="ERSSTv5 (ref)")
    for row in stats_rows:
        ang = np.arccos(np.clip(row["r"], -1, 1))
        x, y = row["sigma_norm"] * np.cos(ang), row["sigma_norm"] * np.sin(ang)
        ax.scatter(x, y, color=MODEL_COLORS[row["modelo"]], s=55, zorder=6, edgecolor="k", linewidth=0.4)

    ax.set_xlim(0, max_r)
    ax.set_ylim(0, max_r)
    ax.set_aspect("equal")
    ax.set_xlabel("sigma normalizado (r=1)")
    ax.set_ylabel("sigma normalizado (r=0)")
    ax.set_title(title, fontsize=10)


fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, (label, idx_key) in zip(axes, [("ONI (Nino3.4)", "ONI (Nino3.4)"),
                                        ("RONI (Nino3.4)", "RONI (Nino3.4)"),
                                        ("ICEN (Nino1+2)", "ICEN (Nino1+2)")]):
    rows = summary[summary["indice"] == idx_key].to_dict("records")
    plot_taylor_diagram(ax, rows, label)

handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=MODEL_COLORS[m],
                       markeredgecolor="k", markersize=8, label=m) for m in test_models]
handles.append(plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="k", markersize=12, label="ERSSTv5 (ref)"))
fig.legend(handles=handles, loc="lower center", ncol=6, fontsize=7, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig(OUT_FIG / "_test_taylor_diagram.png", dpi=100, bbox_inches="tight")
plt.show()

/tmp/ipykernel_40441/2419947929.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 15. Portrait diagram -- resumen de todos los modelos de una vez

Estilo Gleckler et al. (2008): cada metrica se normaliza contra la mediana multi-modelo ((valor - mediana) / mediana). Colormap divergente centrado en 0 (gris = igual a la mediana, rojo = peor, azul = mejor) -- nunca arcoiris, sigue la convencion de la skill de dataviz para cantidades con signo/polaridad.

In [19]:
def portrait_matrix(models, metric_dicts):
    # metric_dicts: {etiqueta: {modelo: valor}}, ya en 'cuanto mas alto, peor'
    # (ej. |sesgo|, RMSE centrado, 1-r). Devuelve (matriz, etiquetas).
    labels = list(metric_dicts.keys())
    mat = np.zeros((len(models), len(labels)))
    for j, lab in enumerate(labels):
        vals = np.array([metric_dicts[lab][m] for m in models])
        med = np.median(vals)
        mat[:, j] = (vals - med) / med if med != 0 else vals
    return mat, labels


def plot_portrait_diagram(mat, models, labels, title, vmax=2.0):
    # vmax fijo (no np.nanmax(abs(mat))): con la normalizacion relativa
    # a la mediana, un solo modelo con sesgo varias veces mayor al resto
    # (ver AWI-CM-1-1-MR: +7.29 con Nino3.4, mediana ~0.17) satura el
    # rango completo y deja todo lo demas invisible. Se recorta en
    # +-2.0 ('el doble o mas de la mediana ya se muestra saturado') --
    # criterio habitual en portrait diagrams (PCMDI/ESMValTool), no una
    # media verdadera del extremo.
    fig, ax = plt.subplots(figsize=(1.3 * len(labels) + 2.5, 0.4 * len(models) + 1.5))
    mat_clipped = np.clip(mat, -vmax, vmax)
    im = ax.imshow(mat_clipped, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='left', fontsize=8)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels(models, fontsize=7)
    for i in range(len(models)):
        for j in range(len(labels)):
            label = f'{mat[i, j]:+.2f}' + ('*' if abs(mat[i, j]) > vmax else '')
            ax.text(j, i, label, ha='center', va='center', fontsize=6,
                    color='white' if abs(mat_clipped[i, j]) > vmax * 0.6 else 'black')
    cbar = plt.colorbar(im, ax=ax, shrink=0.7, extend='both')
    cbar.set_label('Error relativo a la mediana multi-modelo (saturado en +-%.1f)' % vmax)
    ax.set_title(title + '\n(* = valor real fuera de la escala mostrada)', pad=35, fontsize=10)
    plt.tight_layout()
    return fig, ax


metric_dicts = {
    '|sesgo| Nino3.4': {m: abs(bias[m]['nino34'].mean().item()) for m in test_models},
    '|sesgo| Nino1+2': {m: abs(bias[m]['nino12'].mean().item()) for m in test_models},
    "RMSE' Nino3.4": {m: taylor_stats(anom_model[m]['nino34'], anom_obs['nino34'])['rmse_centrado'] for m in test_models},
    '1-r Nino3.4': {m: 1 - taylor_stats(anom_model[m]['nino34'], anom_obs['nino34'])['r'] for m in test_models},
}

mat, labels = portrait_matrix(test_models, metric_dicts)
plot_portrait_diagram(mat, test_models, labels, 'Portrait diagram -- desempeno relativo a la mediana multi-modelo')
plt.savefig(OUT_FIG / '_test_portrait_diagram.png', dpi=100, bbox_inches='tight')
plt.show()

/tmp/ipykernel_40441/1188290420.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 16. ONI con eventos sombreados + resumen de frecuencia/magnitud

Reutiliza `classify_events` (Seccion 12) para sombrear los episodios sobre la serie y comparar conteo/magnitud modelo vs. obs en barras.

In [20]:
def shade_events(ax, events, color='tab:red', alpha=0.15):
    for start, end, _peak in events:
        ax.axvspan(start, end, color=color, alpha=alpha, zorder=1)


oni_events_obs = classify_events(oni_obs, threshold=0.5, min_consecutive=5)
oni_events_model = {m: classify_events(oni_model[m], threshold=0.5, min_consecutive=5) for m in test_models}

WINDOW = (pd.Timestamp('1990-01-01'), pd.Timestamp('2010-12-31'))

fig, ax = plt.subplots(figsize=(12, 3.5))
# axvspan expande el autoescalado del eje X a la fecha de CUALQUIER
# evento, aunque este fuera de la ventana graficada -- por eso el xlim
# se fija explicito al final, no alcanza con .sel(time=slice(...)) en
# los datos.
shade_events(ax, oni_events_obs, color='0.6', alpha=0.25)
oni_obs.sel(time=slice('1990', '2010')).plot(ax=ax, color='k', lw=1.8, label='ONI obs')
oni_model[test_models[0]].sel(time=slice('1990', '2010')).plot(
    ax=ax, color=MODEL_COLORS[test_models[0]], alpha=0.9, label=f'ONI {test_models[0]} (LS)')
ax.axhline(0.5, color='r', ls='--', lw=0.7)
ax.axhline(-0.5, color='b', ls='--', lw=0.7)
ax.set_xlim(*WINDOW)
ax.legend(fontsize=7)
ax.set_title(f'ONI con eventos calidos sombreados (obs, gris) -- {test_models[0]} vs. ERSSTv5, 1990-2010')
plt.savefig(OUT_FIG / '_test_oni_eventos.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = [len(oni_events_model[m]) for m in test_models]
axes[0].bar(range(len(test_models)), counts, color=[MODEL_COLORS[m] for m in test_models])
axes[0].axhline(len(oni_events_obs), color='k', ls='--', lw=1.2, label=f'ERSSTv5 = {len(oni_events_obs)}')
axes[0].set_xticks(range(len(test_models)))
axes[0].set_xticklabels(test_models, rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('N de eventos ONI >=0.5 (>=5 meses)')
axes[0].legend(fontsize=8)
axes[0].set_title('Frecuencia de eventos calidos extremos')

peaks_obs = [ev[2] for ev in oni_events_obs]
data_box = [peaks_obs] + [[ev[2] for ev in oni_events_model[m]] for m in test_models]
axes[1].boxplot(data_box, tick_labels=['ERSSTv5'] + test_models, showmeans=True)
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].set_ylabel('Magnitud del pico ONI por evento')
axes[1].set_title('Magnitud de eventos calidos extremos')
plt.tight_layout()
plt.savefig(OUT_FIG / '_test_eventos_resumen.png', dpi=100, bbox_inches='tight')
plt.show()

/tmp/ipykernel_40441/2014232919.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_40441/2014232919.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 17. Barras de sigma y correlacion, Nino3.4 vs. Nino1+2 (estilo Kaur et al., 2021)

Complementa el diagrama de Taylor con una vista directa por barras -- mas facil de leer numeros exactos por modelo que el polar. sigma normalizado y r, un panel por metrica, un color por caja.

In [21]:
box_stats = {
    m: {k: taylor_stats(anom_model[m][k], anom_obs[k]) for k in ('nino34', 'nino12')}
    for m in test_models
}

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
x = np.arange(len(test_models))
width = 0.35

sigma34 = [box_stats[m]['nino34']['sigma_norm'] for m in test_models]
sigma12 = [box_stats[m]['nino12']['sigma_norm'] for m in test_models]
axes[0].bar(x - width / 2, sigma34, width, label='Nino 3.4', color='tab:blue')
axes[0].bar(x + width / 2, sigma12, width, label='Nino 1+2', color='tab:orange')
axes[0].axhline(1.0, color='k', ls='--', lw=0.8, label='obs (=1)')
axes[0].set_ylabel('sigma normalizado')
axes[0].set_title('Desviacion estandar normalizada, por caja')
axes[0].legend(fontsize=8)

r34 = [box_stats[m]['nino34']['r'] for m in test_models]
r12 = [box_stats[m]['nino12']['r'] for m in test_models]
axes[1].bar(x - width / 2, r34, width, label='Nino 3.4', color='tab:blue')
axes[1].bar(x + width / 2, r12, width, label='Nino 1+2', color='tab:orange')
axes[1].axhline(0, color='k', lw=0.6)
axes[1].set_ylabel('Correlacion (r)')
axes[1].set_title('Correlacion de Pearson (sobre anomalias), por caja')
axes[1].set_xticks(x)
axes[1].set_xticklabels(test_models, rotation=45, ha='right', fontsize=8)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_FIG / '_test_barras_sigma_r.png', dpi=100, bbox_inches='tight')
plt.show()

/tmp/ipykernel_40441/3748312706.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 18. Mapas de correlacion espacial con zonas significativas (estilo Kaur et al., 2021 / Kumar et al. 2023)

Correlacion punto a punto entre la anomalia de TSM del campo 2D completo (no el promedio de caja) y el indice Nino3.4 del mismo dataset (obs correlaciona contra su propio indice, cada modelo contra el suyo, corregido por LS). Punteado = significativo (p<0.05, test t sobre r). Dominio restringido al Pacifico tropical (no el dominio global completo) para que el mapa sea legible y relevante al ENOS. Colormap divergente (RdBu_r), nunca arcoiris.

In [22]:
from scipy import stats as _stats

PACIFIC = dict(lon1=120, lon2=290, lat1=-20, lat2=20)


def field_anomaly(path, box, varname):
    ds = xr.open_dataset(path)
    da = _normalize_time(_select_box(ds[varname], box))
    clim = da.sel(time=slice(f'{REF_INICIO}', f'{REF_FIN}')).groupby('time.month').mean('time')
    anom = da.groupby('time.month') - clim
    return anom.load()


def lag1_autocorr(da, dim='time'):
    # Por POSICION, no por etiqueta de fecha -- xr.align(a,b) hecho con
    # a=da[:-1], b=da[1:] empareja por valor de coordenada de tiempo, y
    # como casi todas las fechas de a tambien estan en b, terminaba
    # comparando (casi) la serie consigo misma en la MISMA fecha, no
    # con el mes anterior: r1 salia ~1 en todos lados y el n efectivo
    # colapsaba al piso (bug real, detectado al ver el mapa sin ningun
    # punteado). Se opera sobre arrays numpy por indice, sin coordenadas.
    vals = da.values
    a, b = vals[:-1], vals[1:]
    a_mean, b_mean = np.nanmean(a, axis=0), np.nanmean(b, axis=0)
    cov = np.nanmean((a - a_mean) * (b - b_mean), axis=0)
    r1 = cov / (np.nanstd(a, axis=0) * np.nanstd(b, axis=0))
    if da.ndim > 1:
        other_dims = [d for d in da.dims if d != dim]
        coords = {k: v for k, v in da.coords.items() if dim not in v.dims}
        return xr.DataArray(r1, dims=other_dims, coords=coords)
    return float(r1)


def corr_map(anom_field, index_1d):
    # Test de significancia corregido por autocorrelacion temporal
    # (Bretherton et al. 1999): la TSM mensual esta fuertemente
    # autocorrelacionada (una anomalia calida persiste varios meses),
    # asi que tratar los ~1980 meses como observaciones independientes
    # (n crudo) sobreestima donde es significativo -- se detecto esto
    # al revisar el mapa: el punteado cubria casi todo el dominio,
    # incluidas zonas con r casi nulo. Se usa n efectivo:
    #   n_eff = n * (1 - r1_x*r1_y) / (1 + r1_x*r1_y)
    # con r1_x, r1_y la autocorrelacion lag-1 del indice y del campo.
    m_field, m_index = xr.align(anom_field, index_1d, join='inner')
    r = xr.corr(m_field, m_index, dim='time')
    n = int(m_index.sizes['time'])
    r1_index = float(lag1_autocorr(m_index))
    r1_field = lag1_autocorr(m_field)
    n_eff = (n * (1 - r1_index * r1_field) / (1 + r1_index * r1_field)).clip(min=3, max=n)
    with np.errstate(invalid='ignore', divide='ignore'):
        t = r.values * np.sqrt((n_eff.values - 2) / (1 - r.values ** 2))
        p = 2 * (1 - _stats.t.cdf(np.abs(t), df=n_eff.values - 2))
    return r, xr.DataArray(p, coords=r.coords, dims=r.dims), n_eff


# El indice usado como regresor es la anomalia propia de cada dataset
# (no la ONI suavizada a 3 meses) -- correlacion punto a punto mensual
# directa. No hace falta pasar por la serie corregida por LS: la
# correlacion de Pearson es invariante a un corrimiento aditivo
# constante por mes calendario (mismo argumento que en la Seccion 9),
# asi que corregir antes o no da el mismo mapa.
anom_field_obs = field_anomaly(obs_path, PACIFIC, 'sst')
r_obs, p_obs, neff_obs = corr_map(anom_field_obs, anom_obs['nino34'])

r_model, p_model = {}, {}
for m in test_models:
    af = field_anomaly(model_path(m), PACIFIC, 'tos')
    r_model[m], p_model[m], _ = corr_map(af, anom_model[m]['nino34'])
    del af  # liberar el campo 2D crudo, solo se necesitaba para el mapa de r/p

print(f"n crudo=1980 vs. n efectivo (obs): mediana={float(neff_obs.median()):.0f}, "
      f"min={float(neff_obs.min()):.0f}, max={float(neff_obs.max()):.0f}")


def plot_corr_map(ax, r, p, title):
    im = ax.pcolormesh(r.lon, r.lat, r.values, cmap='RdBu_r', vmin=-1, vmax=1, shading='auto')
    lon2d, lat2d = np.meshgrid(r.lon.values, r.lat.values)
    sig = p.values < 0.05
    step = 4
    ax.scatter(lon2d[::step, ::step][sig[::step, ::step]], lat2d[::step, ::step][sig[::step, ::step]],
               s=1.5, color='k', alpha=0.5, zorder=3)
    ax.set_title(title, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    return im


n_panels = len(test_models) + 1
ncols = 4
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.3 * nrows))
axes = np.array(axes).reshape(-1)

im = plot_corr_map(axes[0], r_obs, p_obs, 'ERSSTv5 (obs)')
for i, m in enumerate(test_models, start=1):
    plot_corr_map(axes[i], r_model[m], p_model[m], m)
for j in range(n_panels, len(axes)):
    axes[j].axis('off')

fig.colorbar(im, ax=axes.tolist(), shrink=0.6, label='r con el indice Nino3.4 propio')
fig.suptitle('Correlacion espacial de la anomalia de TSM con Nino3.4 (punteado = p<0.05, n efectivo)', y=1.02)
plt.savefig(OUT_FIG / '_test_mapas_correlacion.png', dpi=100, bbox_inches='tight')
plt.show()

/tmp/ipykernel_40441/2731707716.py:24: RuntimeWarning: Mean of empty slice
  a_mean, b_mean = np.nanmean(a, axis=0), np.nanmean(b, axis=0)
/tmp/ipykernel_40441/2731707716.py:25: RuntimeWarning: Mean of empty slice
  cov = np.nanmean((a - a_mean) * (b - b_mean), axis=0)
/home/jonathan/miniconda3/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_40441/2731707716.py:24: RuntimeWarning: Mean of empty slice
  a_mean, b_mean = np.nanmean(a, axis=0), np.nanmean(b, axis=0)
/tmp/ipykernel_40441/2731707716.py:25: RuntimeWarning: Mean of empty slice
  cov = np.nanmean((a - a_mean) * (b - b_mean), axis=0)
/home/jonathan/miniconda3/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_40441/2731707716.py:24: RuntimeWarning: Mean 

n crudo=1980 vs. n efectivo (obs): mediana=216, min=124, max=695


/tmp/ipykernel_40441/2731707716.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
